In [2]:
from datasets import load_dataset
from PIL import Image
import os

print("라이브러리 OK")

라이브러리 OK


In [5]:
save_dir = "../data/openfake"

os.makedirs(f"{save_dir}/real", exist_ok=True)
os.makedirs(f"{save_dir}/fake", exist_ok=True)

print("폴더 생성 완료")
print(os.listdir("../data"))

폴더 생성 완료
['.ipynb_checkpoints', 'cifake-real-and-ai-generated-synthetic-images.zip', 'openfake', 'test', 'train']


In [6]:
from io import BytesIO
from PIL import Image as PILImage
from datasets.features import Image as DatasetImage

print("OpenFake 스트리밍 연결 중...")
ds = load_dataset("ComplexDataLab/OpenFake", split="train", streaming=True).cast_column("image", DatasetImage(decode=False))
print("연결 완료!")

OpenFake 스트리밍 연결 중...


Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/206 [00:00<?, ?it/s]

연결 완료!


In [ ]:
TARGET_PER_MODEL = 3000  # 기존 1000 → 3000 (추가 2000장씩)
TARGET_REAL = 50000      # 기존 20000 → 50000 (추가 30000장)

# 이미 저장된 파일 카운트 (이어받기)
real_count = len(os.listdir(f"{save_dir}/real"))
fake_counts = {}
for fname in os.listdir(f"{save_dir}/fake"):
    model_name = "_".join(fname.split("_")[:-1])
    fake_counts[model_name] = fake_counts.get(model_name, 0) + 1

print(f"이어받기 시작 - 기존 real={real_count}장, fake={sum(fake_counts.values())}장")
print(f"fake 모델별: {fake_counts}\n")

total_saved = 0

for item in ds:
    try:
        label = item["label"]
        model = item.get("model", "unknown")
        img_bytes = item["image"]["bytes"]

        if img_bytes is None:
            continue

        image = PILImage.open(BytesIO(img_bytes))
        image.load()

        if image.mode != "RGB":
            image = image.convert("RGB")

        if label == "fake":
            if fake_counts.get(model, 0) >= TARGET_PER_MODEL:
                continue
            fake_counts[model] = fake_counts.get(model, 0) + 1
            save_path = f"{save_dir}/fake/{model}_{fake_counts[model]:05d}.jpg"
            image.save(save_path, "JPEG", quality=90)

        elif label == "real":
            if real_count >= TARGET_REAL:
                continue
            real_count += 1
            save_path = f"{save_dir}/real/real_{real_count:06d}.jpg"
            image.save(save_path, "JPEG", quality=90)

        total_saved += 1

        if total_saved % 1000 == 0:
            print(f"[+{total_saved}장 추가] real={real_count} | fake={sum(fake_counts.values())}장 | 모델 수={len(fake_counts)}")
            print(f"  fake 모델별: {fake_counts}\n")

    except Exception as e:
        print(f"스킵: {e}")
        continue

    real_done = real_count >= TARGET_REAL
    fake_done = all(v >= TARGET_PER_MODEL for v in fake_counts.values()) and len(fake_counts) >= 15
    if real_done and fake_done:
        print("목표 달성! 완료")
        break

print(f"\n최종: real={real_count}장, fake={sum(fake_counts.values())}장")
print(f"fake 모델별: {fake_counts}")

이어받기 시작 - 기존 real=50000장, fake=70306장
fake 모델별: {'chroma': 1496, 'dalle-3': 3000, 'flux-1.1-pro': 3000, 'flux-amateursnapshotphotos': 1373, 'flux-mvc5000': 3000, 'flux-realism': 430, 'flux.1-dev': 3000, 'flux.1-schnell': 3000, 'gpt-image-1': 3000, 'grok-2-image-1212': 2152, 'hidream-i1-full': 3000, 'ideogram-3.0': 3000, 'imagen-3.0-002': 1403, 'imagen-4.0': 2378, 'midjourney-6': 3000, 'midjourney-7': 1074, 'mystic': 3000, 'sd-1.5-dreamshaper': 3000, 'sd-1.5-epicdream': 3000, 'sd-1.5': 3000, 'sd-2.1': 3000, 'sd-3.5': 3000, 'sdxl-epic-realism': 3000, 'sdxl-juggernaut': 3000, 'sdxl-realvis-v5': 3000, 'sdxl-touchofrealism': 3000, 'sdxl': 3000}

스킵: cannot identify image file <_io.BytesIO object at 0x000001D000442A20>
[+1000장 추가] real=50000 | fake=71306장 | 모델 수=27
  fake 모델별: {'chroma': 1613, 'dalle-3': 3000, 'flux-1.1-pro': 3000, 'flux-amateursnapshotphotos': 1469, 'flux-mvc5000': 3000, 'flux-realism': 457, 'flux.1-dev': 3000, 'flux.1-schnell': 3000, 'gpt-image-1': 3000, 'grok-2-image-1212

C:\Users\user\miniconda3\envs\deepfake\lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))


In [1]:
import os

real_count = len(os.listdir("../data/openfake/real"))
fake_files = os.listdir("../data/openfake/fake")
fake_counts = {}
for fname in fake_files:
    model_name = "_".join(fname.split("_")[:-1])
    fake_counts[model_name] = fake_counts.get(model_name, 0) + 1

print(f"real: {real_count}장")
print(f"fake: {len(fake_files)}장")
print(f"모델 수: {len(fake_counts)}개")
print()
for model, count in sorted(fake_counts.items()):
    status = "✅" if count >= 2000 else "⚠️"
    print(f"  {status} {model}: {count}장")

real: 50000장
fake: 78444장
모델 수: 27개

  ✅ chroma: 3000장
  ✅ dalle-3: 3000장
  ✅ flux-1.1-pro: 3000장
  ✅ flux-amateursnapshotphotos: 3000장
  ✅ flux-mvc5000: 3000장
  ⚠️ flux-realism: 967장
  ✅ flux.1-dev: 3000장
  ✅ flux.1-schnell: 3000장
  ✅ gpt-image-1: 3000장
  ✅ grok-2-image-1212: 3000장
  ✅ hidream-i1-full: 3000장
  ✅ ideogram-3.0: 3000장
  ✅ imagen-3.0-002: 3000장
  ✅ imagen-4.0: 3000장
  ✅ midjourney-6: 3000장
  ✅ midjourney-7: 2477장
  ✅ mystic: 3000장
  ✅ sd-1.5: 3000장
  ✅ sd-1.5-dreamshaper: 3000장
  ✅ sd-1.5-epicdream: 3000장
  ✅ sd-2.1: 3000장
  ✅ sd-3.5: 3000장
  ✅ sdxl: 3000장
  ✅ sdxl-epic-realism: 3000장
  ✅ sdxl-juggernaut: 3000장
  ✅ sdxl-realvis-v5: 3000장
  ✅ sdxl-touchofrealism: 3000장
